# Distilled Tiny RNNs — first Colab pilot

**Purpose:** verify the pipeline on your GPU and measure its runtime. This is a short synthetic first-stage, parametric rare-strategy pilot. It is not the proposal's mechanistic gate, a convergence study, or a reproduction of the published method.

Open this notebook in a fresh Colab session and select **Runtime → Change runtime type → T4 GPU**. The first cell downloads a fixed public revision without a GitHub token. Mount your own Google Drive to save results and checkpoints, then run the cells in order.

The notebook runs verification tests before training.

The pilot trains 1,000 synthetic people with 100 trials each. It uses 100 teacher steps and 150 steps per student arm, with a 10-minute limit per phase checked between steps. A phase that hits its time limit is saved as incomplete and resumes when rerun. Setup, checkpoints, and evaluation add time beyond training caps.

Keep the same notebook revision and run name to resume after a disconnect. Use a new run name when settings or code change. Use one runtime per output folder. Disconnect the GPU runtime when finished.


In [ ]:
"""Download one pinned public GitHub revision for a Colab launcher.

This standard-library module is embedded into the launchers. Each invocation
installs into a fresh runtime-local destination; saved runs belong elsewhere.
"""
from pathlib import Path, PurePosixPath
import io
import os
import re
import shutil
import stat
import tempfile
import urllib.error
import urllib.parse
import urllib.request
import zipfile


_ARCHIVE_LIMIT = 50 * 1024 * 1024
_EXTRACTED_LIMIT = 100 * 1024 * 1024


class _GitHubRedirectHandler(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, req, fp, code, msg, headers, newurl):
        parsed = urllib.parse.urlsplit(newurl)
        if (parsed.scheme != "https" or parsed.hostname != "codeload.github.com"
                or parsed.username is not None or parsed.password is not None
                or parsed.port not in (None, 443)):
            raise RuntimeError("GitHub returned an unexpected download destination")
        return super().redirect_request(req, fp, code, msg, headers, newurl)


def _extract_project_archive(payload, destination):
    """Validate all archive paths before extracting a single GitHub root."""
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    if os.path.lexists(destination):
        raise ValueError("Use a fresh empty destination path for the project")
    with zipfile.ZipFile(io.BytesIO(payload)) as archive:
        entries = archive.infolist()
        if not entries or len(entries) > 10000:
            raise ValueError("Invalid project archive entry count")
        roots, seen, planned = set(), set(), []
        total_size = 0
        for entry in entries:
            name = entry.filename
            components = name.rstrip("/").split("/")
            mode = stat.S_IFMT(entry.external_attr >> 16)
            if (name.startswith("/") or "\\" in name or any(p in ("", ".", "..") for p in components)
                    or mode not in (0, stat.S_IFREG, stat.S_IFDIR) or entry.flag_bits & 1):
                raise ValueError("Unsafe project archive entry")
            roots.add(components[0])
            relative = PurePosixPath(*components[1:])
            normalized = str(relative)
            if normalized in seen:
                raise ValueError("Duplicate project archive entry")
            seen.add(normalized)
            if len(components) == 1:
                if not entry.is_dir():
                    raise ValueError("Project archive must contain a root directory")
                continue
            total_size += entry.file_size
            if total_size > _EXTRACTED_LIMIT:
                raise ValueError("Project archive is too large")
            planned.append((entry, relative))
        if len(roots) != 1 or not planned:
            raise ValueError("Project archive must have one root directory")
        files = {str(path) for entry, path in planned if not entry.is_dir()}
        for _, path in planned:
            if any(str(parent) in files for parent in path.parents):
                raise ValueError("Conflicting project archive paths")
        staging = Path(tempfile.mkdtemp(prefix=".colab-project-", dir=destination.parent))
        try:
            for entry, relative in planned:
                target = staging.joinpath(*relative.parts)
                if entry.is_dir():
                    target.mkdir(parents=True, exist_ok=True)
                else:
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with archive.open(entry) as source, target.open("xb") as output:
                        shutil.copyfileobj(source, output, length=1024 * 1024)
            # Caller selects an unused path. Recheck to avoid replacing a path
            # that appeared while downloading/extracting.
            if os.path.lexists(destination):
                raise ValueError("Project destination already exists")
            staging.rename(destination)
        finally:
            if staging.exists():
                shutil.rmtree(staging)
    return destination


def fetch_public_project(repo, commit, destination):
    """Fetch and safely extract a public ZIP pinned to a full Git commit."""
    if (not isinstance(repo, str)
            or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9-]{0,38}/[A-Za-z0-9_.-]{1,100}", repo)
            or repo.rsplit("/", 1)[-1] in {".", ".."}):
        raise ValueError("Repository must be an owner/name pair")
    if not isinstance(commit, str) or not re.fullmatch(r"[0-9a-fA-F]{40}", commit):
        raise ValueError("A full 40-character Git commit is required")
    if os.path.lexists(destination):
        raise ValueError("Use a fresh destination path for the project")
    request = urllib.request.Request(
        f"https://github.com/{repo}/archive/{commit}.zip",
        headers={"User-Agent": "distilled-rnns-colab"},
    )
    try:
        opener = urllib.request.build_opener(_GitHubRedirectHandler())
        with opener.open(request, timeout=60) as response:
            payload = response.read(_ARCHIVE_LIMIT + 1)
    except urllib.error.HTTPError as error:
        if error.code == 404:
            raise RuntimeError(
                "GitHub could not find the pinned revision. Check that the repository is public."
            ) from None
        raise RuntimeError(f"GitHub project download failed (HTTP {error.code}).") from None
    except (OSError, RuntimeError, ValueError):
        raise RuntimeError("GitHub project download failed. Check your connection and try again.") from None
    if len(payload) > _ARCHIVE_LIMIT:
        raise RuntimeError("The project download exceeds the size limit")
    try:
        return _extract_project_archive(payload, destination)
    except Exception:
        raise RuntimeError("The downloaded project archive could not be safely installed") from None


from pathlib import Path
import os, sys, hashlib, json, subprocess, tempfile
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

EXPECTED_SOURCE_SHA256 = "e161fb5f9f7215c05d7fa1bfe22edf3225809c5c8258013e75fa2be051b95436"
GITHUB_REPOSITORY = "WeaversMask/distilled-tiny-rnns-benchmark"
PINNED_COMMIT = "6ef9574f72c1a17b6f219aa77d3cdc2d6382aa00"
if IN_COLAB:
    destination = Path(tempfile.mkdtemp(prefix="distilled-source-")) / "project"
    PROJECT = fetch_public_project(GITHUB_REPOSITORY, PINNED_COMMIT, destination)
    drive.mount("/content/drive")
    DRIVE_PROJECT = Path("/content/drive/MyDrive/distilled-tiny-rnns")
    RUNS = DRIVE_PROJECT / "runs"
else:
    PROJECT = Path(os.environ.get("DISTILLED_PROJECT_ROOT", Path.cwd())).resolve()
    if not (PROJECT / "pyproject.toml").exists() and (PROJECT.parent / "pyproject.toml").exists():
        PROJECT = PROJECT.parent
    RUNS = Path(os.environ.get("DISTILLED_OUTPUT_ROOT", str(PROJECT / "outputs/notebook")))

digest = hashlib.sha256()
for path in sorted((PROJECT / "src/distilled_rnns").glob("*.py")):
    digest.update(path.name.encode())
    digest.update(path.read_bytes())
assert digest.hexdigest() == EXPECTED_SOURCE_SHA256, "Downloaded source differs from the verified notebook revision."
os.chdir(PROJECT)
RUNS.mkdir(parents=True, exist_ok=True)
print("Project:", PROJECT)
print("Saved runs:", RUNS)


## Install and verify

Installation keeps a compatible preinstalled PyTorch version. Exact installed versions are saved with every run. Tests cover the simulator, causal inputs, test masks, student independence, standard-GRU agreement, and checkpoint resume. GPU-specific checks run when CUDA is present; local CPU verification cannot replace these.


In [ ]:
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", ".[test]"], check=True)
# Editable installs create a startup path file; this kernel is already running.
# Activate the verified source immediately in both Colab and local kernels.
import importlib
source_dir = str((PROJECT / "src").resolve())
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)
importlib.invalidate_caches()
from distilled_rnns.config import ExperimentConfig
print("Project import: ready")
import torch
print("PyTorch:", torch.__version__)
if IN_COLAB and not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime before continuing; this notebook will not silently run the Colab pilot on CPU.")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "CPU (local verification only)")
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], check=True)


## Small smoke test

Twenty people and four training steps per model check that your runtime can complete the whole pipeline. The test runs on the selected GPU in Colab, saves its outputs, and verifies that a second invocation resumes successfully.


In [ ]:
from distilled_rnns.config import ExperimentConfig
from distilled_rnns.experiment import run_experiment

SMOKE_RUN = RUNS / ("smoke-" + EXPECTED_SOURCE_SHA256[:12])
smoke_config = ExperimentConfig.from_file(PROJECT / "configs/smoke.json")
smoke = run_experiment(smoke_config, SMOKE_RUN, device=DEVICE)
assert smoke["status"] == "complete", smoke
smoke_resumed = run_experiment(smoke_config, SMOKE_RUN, device=DEVICE)
assert smoke_resumed["per_subject"] == smoke["per_subject"], "Resumed predictions differ"
print("Smoke test and resume passed.")


## Bounded pilot

The default configuration is in `configs/pilot.json`. It has a 2% rare group whose model-based weight is zero; the majority uses hybrid strategies. The teacher pools everyone; each student has independent weights. Solo and distilled students start from identical weights. Validation chooses each student's weights separately.

The notebook's short runs may be underfit. Read the losses and timings before deciding how long to train the scientific experiment. **Do not infer an effect or a null from this pilot.**


In [ ]:
RUN_NAME = "pilot-v1"  # Keep for resume; change for a new configuration or source revision.
CONFIG_NAME = "pilot" if IN_COLAB else os.environ.get("DISTILLED_NOTEBOOK_CONFIG", "smoke")
config = ExperimentConfig.from_file(PROJECT / "configs" / f"{CONFIG_NAME}.json")
print(json.dumps(config.to_dict(), indent=2))
report = run_experiment(config, RUNS / RUN_NAME, device=DEVICE)
if report["status"] != "complete":
    print(report["message"])


## Results and next decision

NLL measures prediction of held-out first-stage choices; lower is better. KL measures disagreement with the known generating policy on the same histories; lower is better. Diagnostic scores include only trials where true and reference policies disagree by more than 0.2, and are `null` if no such test trials exist.

On the plot, positive x means distillation predicts better than solo training; positive y means its policy fidelity is worse. These are descriptive points from one short run. They do not establish statistical significance or strategy recovery on new histories.

The timing report converts training wall time at your rate only if every session in this run used a T4. Mixed-device histories receive no cost estimate. It excludes setup, evaluation, and idle billing. Before expanding the experiment, inspect convergence, GPU memory, diagnostic support, and actual Colab unit consumption. The full grid and scientific gate require additional implementations and validation described in `docs/roadmap.md`.


In [ ]:
if report["status"] == "complete":
    from IPython.display import display, Image
    print(json.dumps({"groups": report["groups"], "timing": report["timing"]}, indent=2))
    display(Image(filename=str(Path(report["output_dir"]) / "quadrant.png")))
    print("Saved:", report["output_dir"])
    print("Keep the entire run folder, including report.json, manifest.json, training.json, data.npz, predictions.npz, environments/, and the three .pt checkpoints.")
else:
    print("Pilot incomplete; resume before interpreting metrics.")
